# Expressivity across qubits and training

This notebook uses the local release candidate containing the Fourier expressivity analyzer. It compares a binary classifier and two policies before and after optimization, then verifies a width sweep through 16 qubits. The analyzer describes the current ideal circuit's finite Fourier series, not a global model-capacity measure or evidence of learning advantage. It uses its own exact statevector evaluation even when the model uses another simulator.

In [ ]:
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt
from qasp import create_classifier, create_policy, expressivity_analyzer, Normal

torch.set_default_dtype(torch.float64)
torch.manual_seed(31)


## Create models from configuration

One scalar feature is encoded on the first qubit. Two uploads use independently trainable scales. The classifier returns an expectation score; the softmax policy returns logits; the Born policy returns action probabilities. Changing input scales can move frequency locations, while changing only variational angles changes coefficients within the fixed allowed support.

In [ ]:
def make_model(kind):
    config = {
        "feature_map": "AngleEmbedding",
        "feature_map_config": {"features": 1, "rotations": ["X 0"], "input_scaling": True},
        "ansatz": "AngleAnsatz",
        "ansatz_config": {"n_layers": 2, "reuploading": True,
                          "rotations": ["X 0", "X 1", "XX 0 1"], "include_imprimitive": False},
        "init": Normal(0, 0.3), "input_scaling_init": "ones",
        "device": "default.qubit", "diff_method": "backprop",
    }
    if kind == "classifier":
        return create_classifier({**config, "readout": "expectation", "observables": ["Z 0"]}, 2, 2).double()
    if kind == "softmax":
        return create_policy("PQCSoftmax", {**config, "n_actions": 2,
                            "observables": ["Z 0", "Z 1"], "fc_layer": True}, 2).double()
    return create_policy("PQCBorn", {**config, "n_actions": 2, "policy": "parity-like",
                        "training_method": "expectation", "parity_cnot_decomposition": False}, 2).double()

models = {kind: make_model(kind) for kind in ["classifier", "softmax", "born"]}
x = torch.linspace(-1.5, 1.5, 16).reshape(-1, 1)
y = (x[:, 0] > 0).long()
probe = torch.linspace(-1.7, 1.7, 53).reshape(-1, 1)
def outputs(model, data):
    return torch.stack([model(row).reshape(-1) for row in data])


## Save the initial spectrum, train, then analyze again

The classifier minimizes logistic loss. Policies maximize the exact expected reward of a contextual bandit: action 1 is rewarded for positive inputs and action 0 for negative inputs. These short, reproducible learning experiments do not claim convergence or policy advantage. Direct outputs at off-grid inputs validate both spectra. Softmax is nonlinear, so reconstruct logits first, then apply softmax if probabilities are needed.

In [ ]:
comparisons = {}
for kind, model in models.items():
    initial = expressivity_analyzer(model)
    np.testing.assert_allclose(initial.evaluate(probe), outputs(model, probe).detach(), atol=1e-9)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.04)
    losses = []
    for epoch in range(30):
        optimizer.zero_grad()
        values = outputs(model, x)
        if kind == "classifier":
            loss = torch.nn.functional.softplus(-(2*y-1)*values[:, 0]).mean()
        else:
            probabilities = values.softmax(-1) if kind == "softmax" else values
            loss = -probabilities[torch.arange(len(x)), y].mean()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    final = expressivity_analyzer(model)
    np.testing.assert_allclose(final.evaluate(probe), outputs(model, probe).detach(), atol=1e-9)
    comparisons[kind] = (initial, final, losses)
    print(kind, "objective:", round(losses[0], 4), "→", round(losses[-1], 4),
          "active:", initial.active_counts, "→", final.active_counts)


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 10), constrained_layout=True)
for row, (kind, (initial, final, losses)) in enumerate(comparisons.items()):
    axes[row, 0].plot(losses)
    axes[row, 0].set(title=f"{kind}: training objective", xlabel="Epoch")
    for spectrum, label, marker in [(initial, "Before", "o"), (final, "After", "x")]:
        axes[row, 1].scatter(spectrum.frequencies[:, 0], spectrum.magnitudes[:, 0], label=label, marker=marker)
        axes[row, 2].plot(probe[:, 0], spectrum.evaluate(probe)[:, 0], label=label)
    axes[row, 1].set(title="First output: coefficient magnitudes", xlabel="Frequency", ylabel="|coefficient|")
    axes[row, 2].set(title="First output: reconstruction", xlabel="Input")
    axes[row, 1].legend(); axes[row, 2].legend()
plt.show()


## Qubit sweep with an independent closed-form reference

All qubits receive the same RY(x) encoding. Local Z is cos(x); full-register parity is cos(x)^n. Thus the allowed count is 2n+1, local Z has two active frequencies, and parity has n+1. The local and parity readouts differ even for the same circuit. This structured family has a small phase grid; arbitrary independent features or learned projection directions can make analysis exponential. The 16-qubit example is not a general scaling guarantee.

In [ ]:
def width_model(n):
    return create_classifier({
        "feature_map": "NeuralEmbedding",
        "feature_map_config": {"features": 1, "rotations": "Y"},
        "ansatz": "AngleAnsatz", "ansatz_config": {"n_layers": 1, "rotations": "Z"},
        "init": "zeros", "input_scaling_init": "ones", "readout": "expectation",
    }, n, 2).double()

widths = [2, 4, 6, 8, 10, 12, 14, 16]
sweep = expressivity_analyzer(model_factory=width_model, qubit_list=widths,
                            observables=lambda n: ["Z 0", "Z" * n])
for n, spectrum in sweep.results_by_qubit.items():
    z = np.array([[0.37], [0.81]])
    np.testing.assert_allclose(spectrum.evaluate(z),
        np.column_stack([np.cos(z[:, 0]), np.cos(z[:, 0]) ** n]), atol=1e-10)
active = np.array([s.active_counts for s in sweep.results_by_qubit.values()])
plt.plot(widths, sweep.frequency_counts, "o-", label="Allowed")
plt.plot(widths, active[:, 0], "s-", label="Active: local Z")
plt.plot(widths, active[:, 1], "^-", label="Active: parity")
plt.xlabel("Qubits"); plt.ylabel("Signed frequency count"); plt.legend(); plt.show()


## Interpretation

Snapshots remain valid after the model changes. Fixed unscaled encodings keep their allowed support; learned input scales or neural projections may move the support. Active counts depend on the numerical threshold and can increase without improving prediction. Compare coefficients, non-DC power, model performance and resource cost together. The regression suite separately checks exact backend reconstructions, preservation of model state/gradients, complex conjugate symmetry, guardrails and independent algebraic references. Approximate/noisy backend outputs need separate error tolerances.